# Intro


Live et exercices créés par Phuong Nguyen:

[Linkedin](https://www.linkedin.com/in/fphuongnguyen/)

[Newsletter](https://sundayselectstar.substack.com/)

Pour monter en compétences en SQL et résoudre des problèmes business concrets:
https://accelerateur-data.systeme.io/accelerateur-sql

# Téléchargement du dataset dans DuckDB

Lien du dataset: https://www.kaggle.com/datasets/bhanuthakurr/cleaned-contoso-dataset/data

Exécutez toutes les cellules dans cette partie pour un setup complet.



In [ ]:
%pip install duckdb-engine --quiet

In [ ]:
import sys
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
%matplotlib inline

if 'google.colab' in sys.modules:
    !sudo apt-get update -qq > /dev/null 2>&1
    !pip install -q duckdb > /dev/null 2>&1
    !pip install -q kagglehub > /dev/null 2>&1
    !pip install -q jupysql > /dev/null 2>&1

# Load SQL extension
%load_ext sql

# Config
%config SqlMagic.autopandas = True
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
import duckdb
import kagglehub
from pathlib import Path
import time

# Télécharger depuis Kaggle
print("📥 Téléchargement du dataset Kaggle...")
start_time = time.time()
dataset_path = kagglehub.dataset_download(
    "bhanuthakurr/cleaned-contoso-dataset"
)
download_time = time.time() - start_time
print(f"✓ Dataset téléchargé en {download_time:.2f}s\n")

# Créer une connexion DuckDB persistante
print("🔌 Création de la base DuckDB...")
con = duckdb.connect('accelerateur-sql.duckdb')
print("✓ Connecté!\n")

# Charger les fichiers CSV
print("📂 Chargement des données...")
csv_files = list(Path(dataset_path).glob('*.csv'))
print(f"📁 {len(csv_files)} fichiers CSV trouvés\n")

total_rows = 0
total_start = time.time()

for idx, csv_file in enumerate(csv_files, 1):
    table_name = csv_file.stem

    print(f"[{idx}/{len(csv_files)}] '{table_name}'...", end=" ", flush=True)

    # Charger directement avec DuckDB
    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv_auto('{csv_file}')
    """)

    # Compter les lignes
    row_count = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    col_count = len(con.execute(f"SELECT * FROM {table_name} LIMIT 1").description)

    print(f"✓ ({row_count:,} lignes, {col_count} colonnes)")
    total_rows += row_count

total_time = time.time() - total_start

print("\n" + "=" * 60)
print(f"✓ COMPLET!")
print(f"  • Tables créées: {len(csv_files)}")
print(f"  • Lignes totales: {total_rows:,}")
print(f"  • Temps total: {total_time:.2f}s")
print("=" * 60)

In [ ]:
%%sql con

In [ ]:
%%sql
-- tester la ligne magique
SELECT * FROM FactOnlineSales
LIMIT 5

#Exercices

In [ ]:
%%sql

-- ventes 2e semestre 2007

SELECT
  SUM(SalesAmount) AS total_sales_amount
FROM FactOnlineSales
WHERE DateKey >= '2007-07-01' AND DateKey <= '2007-12-31'

In [ ]:
%%sql



SELECT
  SUM(SalesAmount) AS total_sales_amount
FROM FactOnlineSales
WHERE (DateKey >= '2007-07-01' AND DateKey <= '2007-12-31')
OR ProductKey = 782

In [ ]:
%%sql

SELECT
  SUM(SalesAmount) As total_sales_amount
FROM FactOnlineSales
WHERE DateKey < '2007-07-01' OR DateKey > '2007-12-31'
AND ProductKey = 782

In [ ]:
%%sql

SELECT
  SUM(SalesAmount) As total_sales_amount
FROM FactOnlineSales
WHERE (DateKey < '2007-07-01' OR DateKey > '2007-12-31')
AND ProductKey = 782

In [ ]:
%%sql

SELECT
  SUM(SalesAmount) AS total_sales_amount
FROM FactOnlineSales
WHERE ProductKey = 782
AND DateKey NOT BETWEEN '2007-07-01' AND '2007-12-31'

## Exercices intermédiaires

In [ ]:
%%sql
--vente par magasin

SELECT
  StoreName,
  SUM(SalesAmount) AS total_sales_amount
FROM FactOnlineSales
INNER JOIN DimStore
-- ON FactOnlineSales.StoreKey = DimStore.StoreKey
USING(StoreKey)
GROUP BY StoreName
ORDER BY total_sales_amount DESC

In [ ]:
%%sql

SELECT * FROM DimStore
LIMIT 5

In [ ]:
%%sql

-- vente par zone géo
SELECT
  RegionCountryName,
  SUM(SalesAmount) AS total_sales_amount
FROM FactOnlineSales
INNER JOIN DimStore
-- ON FactOnlineSales.StoreKey = DimStore.StoreKey
USING(StoreKey)
INNER JOIN DimGeography
USING(GeographyKey)
GROUP BY RegionCountryName
ORDER BY total_sales_amount DESC

In [ ]:
%%sql

SELECT * FROM DimGeography
LIMIT 5

## Exercice avancée

In [ ]:
%%sql

SELECT * FROM FactOnlineSales
LIMIT 5

In [ ]:
%%sql
-- les clients qui génèrent le plus de CA



In [ ]:
%%sql

-- identifier les clients qui ont acheté les produits les plus populaires:

-- le plus populaire en CA
-- le plus populaire en quantité

-- les clients qui ont acheté les 10 produits les plus populaires en CA
-- les clients qui ont acheté les 3 produits les plus populaires en CA




In [ ]:
%%sql

-- les clients qui ont acheté les 3 produits les plus populaires en CA

-- 1/ identfier les 3 meilleurs produits en terme de CA
-- 2/ trouver les clients qui ont acheté les 3 meilleurs produits

SELECT
  ProductKey
FROM FactOnlineSales
GROUP BY ProductKey
ORDER BY SUM(SalesAmount) DESC
LIMIT 3

In [ ]:
%%sql
WITH three_best_products AS (
  SELECT
    ProductKey
  FROM FactOnlineSales
  GROUP BY ProductKey
  ORDER BY SUM(SalesAmount) DESC
  LIMIT 3
)

SELECT
  CustomerKey,
  ProductKey
FROM FactOnlineSales
WHERE ProductKey IN (SELECT * FROM three_best_products)
GROUP BY CustomerKey, ProductKey
ORDER BY CustomerKey, ProductKey

In [ ]:
%%sql
WITH three_best_products AS (
  SELECT
    ProductKey
  FROM FactOnlineSales
  GROUP BY ProductKey
  ORDER BY SUM(SalesAmount) DESC
  LIMIT 3
)

, customers_bought_three_products AS (
  -- customers who bought at least one of 3 best products
SELECT
  CustomerKey,
  ProductKey
FROM FactOnlineSales
WHERE ProductKey IN (SELECT * FROM three_best_products)
GROUP BY CustomerKey, ProductKey
ORDER BY CustomerKey, ProductKey
)

SELECT
  CustomerKey,
FROM customers_bought_three_products
GROUP BY CustomerKey
HAVING COUNT(ProductKey) = 3
ORDER BY CustomerKey

In [ ]:
%%sql
WITH three_best_products AS (
  SELECT
    ProductKey
  FROM FactOnlineSales
  GROUP BY ProductKey
  ORDER BY SUM(SalesAmount) DESC
  LIMIT 3
)

, customers_bought_three_products AS (
  -- customers who bought at least one of 3 best products
SELECT
  FactOnlineSales.CustomerKey,
  FactOnlineSales.ProductKey
FROM FactOnlineSales
INNER JOIN three_best_products
ON FactOnlineSales.ProductKey = three_best_products.ProductKey
GROUP BY FactOnlineSales.CustomerKey, FactOnlineSales.ProductKey
ORDER BY FactOnlineSales.CustomerKey
)

SELECT
  CustomerKey,
FROM customers_bought_three_products
GROUP BY CustomerKey
HAVING COUNT(ProductKey) = 3
ORDER BY CustomerKey